In [13]:
import pandas as pd
import numpy as np
import yfinance as yf

In [26]:
tickers = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS', 'HINDUNILVR.NS', 'ITC.NS', 'BAJFINANCE.NS', 'KOTAKBANK.NS', 'LT.NS']

downloaded = yf.download(
    tickers,
    start='2021-08-01',
    end='2026-09-01',
    auto_adjust=False,
    progress=False
)

# Use adjusted prices when available; otherwise use closing prices
price_column = 'Adj Close' if 'Adj Close' in downloaded.columns.get_level_values(0) else 'Close'
downloaded = downloaded[price_column].dropna(how='all')

#format the index in dd/mm/yy format
df = downloaded.copy()
df.index = df.index.strftime('%d/%m/%y 16:00')

#rename the index column name
df.index.name = 'date_gsheets'

#Record column to match exact
df = df[tickers]

# Save to CSV
df.to_csv('10stocks_price.csv')

print("CSV created successfully!")

CSV created successfully!


In [27]:
df = pd.read_csv('10stocks_price.csv')
df.head()

,date_gsheets,RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,HINDUNILVR.NS,ITC.NS,BAJFINANCE.NS,KOTAKBANK.NS,LT.NS
0,02/08/21 16:00,938.672363,2796.687744,1421.035278,665.656128,655.445557,2139.020508,163.225983,604.642822,331.062927,1533.899048
1,03/08/21 16:00,945.579346,2853.587646,1441.633545,671.294312,664.102783,2188.580811,165.273224,617.456360,335.148804,1553.287109
2,04/08/21 16:00,952.826111,2844.075684,1436.451416,685.611938,687.429077,2179.640625,164.249603,620.113586,347.933197,1542.690063
3,05/08/21 16:00,966.640137,2852.762695,1440.196655,694.759338,675.693787,2161.852295,169.485718,609.299255,352.267609,1546.681885
4,06/08/21 16:00,946.168213,2875.218262,1437.278564,698.409058,671.076660,2174.505371,168.698364,606.963623,353.351257,1530.477539


In [31]:
#compute daily return of the stock
type(df['date_gsheets'][0])

str

In [38]:
# Convert the existing date column to datetime
df['date_gsheets'] = pd.to_datetime(
    df['date_gsheets'],
    format='%d/%m/%y %H:%M'
)
type(df['date_gsheets'][0])

pandas.Timestamp

In [42]:
df['date_gsheets'].describe()

count                          1260
mean     2024-02-17 02:41:08.571428
min             2021-08-02 16:00:00
25%             2022-11-09 04:00:00
50%             2024-02-16 04:00:00
75%             2025-05-29 22:00:00
max             2026-08-31 16:00:00
Name: date_gsheets, dtype: object

In [43]:
df.describe()

,date_gsheets,RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,HINDUNILVR.NS,ITC.NS,BAJFINANCE.NS,KOTAKBANK.NS,LT.NS
count,1260,1260.000000,1260.000000,1260.000000,1260.000000,1260.000000,1260.000000,1260.000000,1260.000000,1260.000000,1260.000000
mean,2024-02-17 02:41:08.571428,1267.108614,3150.848136,1433.084075,786.801392,1055.010299,2327.612616,326.585568,762.276194,375.925397,2881.622488
min,2021-08-02 16:00:00,938.672363,1971.788818,985.299988,606.489258,628.848694,1793.425293,161.100037,515.260437,308.113525,1398.799561
25%,2022-11-09 04:00:00,1145.328186,2905.712341,1304.709045,716.246750,840.359711,2204.290833,275.406502,669.647446,353.389122,1937.499756
50%,2024-02-16 04:00:00,1247.463074,3100.783936,1422.859253,771.695618,1010.307709,2342.701904,358.807877,717.904785,369.999359,3252.985229
75%,2025-05-29 22:00:00,1401.022278,3384.014709,1544.525635,830.077744,1301.000732,2437.603882,390.122986,876.172409,392.339684,3590.371887
max,2026-08-31 16:00:00,1584.971802,4230.709961,1901.082886,996.419800,1464.852417,2909.856689,463.949219,1158.800049,452.456299,4375.364258
std,NaN,148.150485,455.772586,189.749467,95.095196,255.874101,177.250067,82.230560,138.501685,28.303230,884.889901


In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date_gsheets   1260 non-null   datetime64[us]
 1   RELIANCE.NS    1260 non-null   float64       
 2   TCS.NS         1260 non-null   float64       
 3   INFY.NS        1260 non-null   float64       
 4   HDFCBANK.NS    1260 non-null   float64       
 5   ICICIBANK.NS   1260 non-null   float64       
 6   HINDUNILVR.NS  1260 non-null   float64       
 7   ITC.NS         1260 non-null   float64       
 8   BAJFINANCE.NS  1260 non-null   float64       
 9   KOTAKBANK.NS   1260 non-null   float64       
 10  LT.NS          1260 non-null   float64       
 11  data_gsheets   1260 non-null   object        
dtypes: datetime64[us](1), float64(10), object(1)
memory usage: 118.3+ KB


In [57]:
# Remove the accidental column, if it exists
df = df.drop(columns=['data_gsheets'], errors='ignore')

# Ensure the date index is correctly named and formatted
if 'date_gsheets' in df.columns:
    df['date_gsheets'] = pd.to_datetime(
        df['date_gsheets'],
        format='%d/%m/%y %H:%M'
    )
    df.set_index('date_gsheets', inplace=True)
else:
    df.index = pd.to_datetime(df.index)
    df.index.name = 'date_gsheets'

df.head()

,RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,HINDUNILVR.NS,ITC.NS,BAJFINANCE.NS,KOTAKBANK.NS,LT.NS
date_gsheets,,,,,,,,,,
2021-08-02 16:00:00,938.672363,2796.687744,1421.035278,665.656128,655.445557,2139.020508,163.225983,604.642822,331.062927,1533.899048
2021-08-03 16:00:00,945.579346,2853.587646,1441.633545,671.294312,664.102783,2188.580811,165.273224,617.456360,335.148804,1553.287109
2021-08-04 16:00:00,952.826111,2844.075684,1436.451416,685.611938,687.429077,2179.640625,164.249603,620.113586,347.933197,1542.690063
2021-08-05 16:00:00,966.640137,2852.762695,1440.196655,694.759338,675.693787,2161.852295,169.485718,609.299255,352.267609,1546.681885
2021-08-06 16:00:00,946.168213,2875.218262,1437.278564,698.409058,671.076660,2174.505371,168.698364,606.963623,353.351257,1530.477539


In [60]:
returns_df = df.pct_change(1)

In [63]:
# calculate weights for each stock
num_of_stocks = 10
weights = [1/num_of_stocks]*num_of_stocks
weights

[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

In [65]:
#Calculate the variance covariance matrix

vcv_matrix = returns_df.cov()
vcv_matrix

,RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,HINDUNILVR.NS,ITC.NS,BAJFINANCE.NS,KOTAKBANK.NS,LT.NS
RELIANCE.NS,0.000197,0.000053,0.000059,0.000068,0.000061,0.000036,0.000054,0.000091,0.000065,0.000092
TCS.NS,0.000053,0.000201,0.000168,0.000043,0.000037,0.000038,0.000032,0.000061,0.000037,0.000066
INFY.NS,0.000059,0.000168,0.000262,0.000053,0.000047,0.000041,0.000037,0.000073,0.000042,0.000077
HDFCBANK.NS,0.000068,0.000043,0.000053,0.000172,0.000083,0.000038,0.000037,0.000100,0.000090,0.000083
ICICIBANK.NS,0.000061,0.000037,0.000047,0.000083,0.000162,0.000032,0.000040,0.000095,0.000083,0.000080
HINDUNILVR.NS,0.000036,0.000038,0.000041,0.000038,0.000032,0.000168,0.000055,0.000052,0.000046,0.000034
ITC.NS,0.000054,0.000032,0.000037,0.000037,0.000040,0.000055,0.000159,0.000060,0.000051,0.000051
BAJFINANCE.NS,0.000091,0.000061,0.000073,0.000100,0.000095,0.000052,0.000060,0.000318,0.000103,0.000101
KOTAKBANK.NS,0.000065,0.000037,0.000042,0.000090,0.000083,0.000046,0.000051,0.000103,0.000208,0.000075
LT.NS,0.000092,0.000066,0.000077,0.000083,0.000080,0.000034,0.000051,0.000101,0.000075,0.000235


In [66]:
var_p = np.dot(np.transpose(weights), np.dot(vcv_matrix, weights))
var_p

np.float64(7.723181430248045e-05)

In [68]:
sd_p = np.sqrt(var_p)
sd_p

np.float64(0.008788163306543664)

In [71]:
sd_p_annual = sd_p* np.sqrt(252)
sd_p_annual

np.float64(0.13950776754082575)

In [72]:
individual_risks = returns_df.std() * np.sqrt(252)
individual_risks

RELIANCE.NS      0.222698
TCS.NS           0.225243
INFY.NS          0.256867
HDFCBANK.NS      0.208439
ICICIBANK.NS     0.201889
HINDUNILVR.NS    0.205827
ITC.NS           0.200068
BAJFINANCE.NS    0.282915
KOTAKBANK.NS     0.228800
LT.NS            0.243522
dtype: float64